<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Index Generation</h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses Microsoft GraphRAG library to construct a GraphRAG index for the synthetically generated dataset:</li>
        <ul>
            <li>Uses openai/gpt-oss-20b at the chat model</li>
            <li>Uses intfloat/e5-mistral-7b-instruct as the embedding model</li>
        </ul>
    </li>
    <li>Stores index in LanceDB database backed by Minio bucket</li>
    </ul>
</div>

In [1]:
##############################################
# Imports
##############################################
from dotenv import load_dotenv
load_dotenv()
import traceback
import tracemalloc
tracemalloc.start()
import nest_asyncio
nest_asyncio.apply()
import logging
logging.basicConfig(level=logging.INFO)

In [2]:
##############################################
# Generate GraphRAG index and store in LanceDB
##############################################

def generate_graphrag_index(codebase_path: str,
                            graphrag_source_path: str) :

    """
    Generates a GraphRAG index from the provided codebase.
    Args:
        codebase_path: The path to the source code
        graphrag_source_path: The source path used by the GraphRAG index configuration
    Returns:
        None
    """

    ##############################################
    # Imports
    ##############################################
    from minio import Minio
    import os
    import lancedb
    import shutil
    import traceback
    import subprocess
    import string
    import tracemalloc
    tracemalloc.start()
    import nest_asyncio
    nest_asyncio.apply()
    import logging
    logging.basicConfig(level=logging.INFO)

    def prepare_settings(template_path: str, output_path: str):
        logging.info("Preparing settings...")

        try:
            with open(template_path) as f:

                content = string.Template(f.read())

            with open(output_path, "w") as f:

                f.write(content.substitute(os.environ))

        except KeyError as keyerr:

          raise ValueError(f"Required environment variable {keyerr} is not set")


    try:
        logging.info("Starting process...")

        settings_config_path = "templates/settings.yaml.in"

        settings_config_path_updated = "templates/settings.yaml"

        graph_rag_config_path = f"{graphrag_source_path}/settings.yaml"
        
        os.makedirs(f"{graphrag_source_path}/input", exist_ok=True)

        os.makedirs(f"{graphrag_source_path}/output", exist_ok=True)

        prepare_settings(settings_config_path, settings_config_path_updated)

        logging.info("Copying source code to GraphRAG directory...")

        shutil.copytree(codebase_path, f"{graphrag_source_path}/input", dirs_exist_ok=True)

        shutil.copytree("prompts", f"{graphrag_source_path}/prompts", dirs_exist_ok=True)

        logging.info("Running index...")
    
        result = subprocess.run(["bash", "graphrag.sh", graphrag_source_path, graph_rag_config_path], capture_output=True, text=True, check=False)
            
        logging.info(f"\nSubprocess output: {result.stdout}")
        
        if result.stderr:
            
            raise Exception(f"Error processing GraphRAG command: {result.stderr}")
        
    except Exception as e:
        
        logging.error(f"Error processing GraphRAG DB: {e}")

        logging.error(traceback.format_exc())

        raise e

In [6]:
##############################################
# Full Pipeline
##############################################
def graphrag_pipeline(codebase_path: str, graphrag_source_path: str) :
    """
    Executes the full pipeline which generates a GraphRAG index for the
    given directory and stores it in remote and local instances of LanceDB.
    """
    
    try:
        
        generate_graphrag_index(codebase_path=codebase_path,
                                graphrag_source_path=graphrag_source_path)

        logging.info("GraphRAG index generation complete.")
    
    except Exception as e:
            
        logging.error(f"Error processing Sample Codebase Index: {e}")
    
        logging.error(traceback.format_exc())
    


### FULL PIPELINE
Execute full pipeline!

In [ ]:
#########################################################################
# Instance Variables
#########################################################################
_CODEBASE_PATH = "target" # your code metadata directory (generated by data_generation_graphrag_pipeline.ipynb)

_GRAPHRAG_SOURCE_PATH = "graph_rag_app/source"

In [ ]:
graphrag_pipeline(codebase_path=_CODEBASE_PATH,
                  graphrag_source_path=_GRAPHRAG_SOURCE_PATH,)